# Muse Waffle v3 — Training Notebook

Run the cells below in order: **Setup → Create/Load Model → Train → Test**.

In [1]:
!pip install -r requirements.txt

  Using cached dotenv-0.9.9-py2.py3-none-any.whl.metadata (279 bytes)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 729.9/729.9 kB 321.9 kB/s eta 0:00:00a 0:00:01
Using cached dotenv-0.9.9-py2.py3-none-any.whl (1.9 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 346.6 kB/s eta 0:00:00a 0:00:01
  Attempting uninstall: pygments
    Found existing installation: Pygments 2.17.2
    Uninstalling Pygments-2.17.2:
      Successfully uninstalled Pygments-2.17.2
  Attempting uninstall: rich
    Found existing installation: rich 13.7.1
    Uninstalling rich-13.7.1:
      Successfully uninstalled rich-13.7.1

[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("/notebooks/"))  # make src/ importable

import torch
from pathlib import Path
import json
from src.dataset import load_cached_paths, download_dataset
from src.model import build_model, build_infill_model, load_model, DEFAULT_CONFIG, append_epoch_to_config
from src.trainer import train
from src.infill_trainer import train_infill, INFILL_WEIGHTS_FILE
from src.generator import generate_and_export, show_text_score
from src.infill_generator import infill_and_export, create_test_partial_xml, load_partial_xml
from src.similarity import compare_to_dataset
from src.tokenizer import (
    identify_empty_measures, score_to_tokens_by_measure, 
    FILL_START, FILL_END, EMPTY_BAR, VOCAB_SIZE
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
print(f"Vocabulary size: {VOCAB_SIZE} (includes infill tokens)")

ModuleNotFoundError: No module named 'torch'

## 1. Create or Load Model

`build_model()` initialises fresh weights and saves `checkpoints/config.json`.  
`load_model()` reloads weights from a previous run — use it after training.  
Edit `cfg` below to change architecture before building.

## 2. Train

Downloads the dataset if not already cached, then runs the training loop.  
Weights are saved to `checkpoints/model.pt` after every epoch.

## 4. Infill Training

Train a model with Fill-in-the-Middle (FIM) capability. This teaches the model to generate
content that fits between a prefix and suffix context, enabling music in-filling.

## 5. Test Infill (Fill Empty Measures)

Test the infill capability by creating a partial XML with empty measures,
then generating content to fill the gaps.

In [2]:
from pathlib import Path

# Create a test partial XML file with empty measures
TEST_XML_PATH = "output/test_partial.xml"
EMPTY_MEASURES = [3, 4, 5]  # measures to leave empty (these will be filled)

Path("output").mkdir(exist_ok=True)
create_test_partial_xml(TEST_XML_PATH, empty_measures=EMPTY_MEASURES)
print(f"Created test file: {TEST_XML_PATH}")
print(f"Empty measures: {EMPTY_MEASURES}")

# Verify gaps are detected
score, gaps = load_partial_xml(TEST_XML_PATH)
print(f"Detected gaps: {gaps}")

Created test file: output/test_partial.xml
Empty measures: [3, 4, 5]
[<music21.stream.Part 0x7f66cc43dd50>]
{1, 2, 3, 4, 5, 6, 7, 8}
<music21.note.Rest whole>, True, True
<music21.note.Rest whole>, True, True
<music21.note.Rest whole>, True, True
Detected gaps: [(3, 5)]


In [3]:
# Run infill generation
INFILL_TEMPERATURE = 0.9
INFILL_TOP_K = 50
INFILL_MAX_NEW = 500

# infill_test_log = []
# def infill_test_log_fn(msg):
#     print(msg)
#     infill_test_log.append(msg)

# print("Running infill generation...")
# print("=" * 50)

# result_paths = infill_and_export(
#     xml_path=TEST_XML_PATH,
#     max_new=INFILL_MAX_NEW,
#     temperature=INFILL_TEMPERATURE,
#     top_k=INFILL_TOP_K,
#     log=infill_test_log_fn,
#     device=DEVICE,
# )

# print("=" * 50)
# print(f"\nOutput files:")
# print(f"  MIDI:     {result_paths['midi']}")
# print(f"  MusicXML: {result_paths['xml']}")

# if result_paths["xml"]:
#     print("\n--- Filled Score Preview ---")
#     show_text_score(result_paths["xml"], infill_test_log_fn)